In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
# =============================================================================
# Punto de entrada del pipeline metadata-driven — Fintech FinPay | Bronze
#
# PARÁMETRO DE ENTRADA (Job Parameter / Widget):
#   source_name = "all"           → procesa todas las fuentes activas del JSON
#   source_name = "transactions"  → procesa SOLO esa fuente
#   source_name = "merchants"     → procesa SOLO esa fuente
#   source_name = "users"         → procesa SOLO esa fuente
#
# Desde un Databricks Job:
#   Task Parameters → Key: source_name | Value: all (o el nombre exacto)
# =============================================================================

In [0]:
import sys
sys.path.append("/Workspace/Users/jean.zelada06@gmail.com/.bundle/bundle_dev_finpay/dev/files/src")

from ingestion_functions import load_archetypes, ingest_source

In [0]:
METADATA_PATH = (
    "/Volumes/fintech_finpay/default/vol_config/ingestion_archetypes.json"
)

2026-05-27 04:05:19,949 [INFO] Received command c on object id p0


In [0]:


SOURCE_FILTER = dbutils.widgets.get("source_name").strip().lower()

print(f"Parámetro recibido → source_name = '{SOURCE_FILTER}'")

2026-05-27 04:05:20,048 [INFO] Received command c on object id p0


Parámetro recibido → source_name = 'all'


In [0]:
# =============================================================================
#
# load_archetypes() ya retorna solo los activos (active == true).
# Luego se aplica el filtro de SOURCE_FILTER:
#   "all"          → se usan todos los activos sin filtro adicional
#   "<source_name>"→ se filtra a UNO; si no existe lanza error controlado
# =============================================================================
all_active = load_archetypes(spark, METADATA_PATH)
 
if SOURCE_FILTER == "all":
    archetypes_to_run = all_active
    print(f"▶ Modo ALL — procesando {len(archetypes_to_run)} fuente(s) activa(s)")
 
else:
    # Filtrar por nombre exacto (case-insensitive)
    archetypes_to_run = [
        a for a in all_active
        if a["source_name"].lower() == SOURCE_FILTER
    ]
 
    if not archetypes_to_run:
        # Construir mensaje de error descriptivo con las fuentes disponibles
        available = [a["source_name"] for a in all_active]
        raise ValueError(
            f"Fuente '{SOURCE_FILTER}' no encontrada o no está activa en el JSON. "
            f"Fuentes activas disponibles: {available}. "
            f"Verifica el parámetro 'source_name' en el Job o que 'active=true' en el JSON."
        )
 
    print(f"▶ Modo SINGLE — procesando exclusivamente: '{archetypes_to_run[0]['source_name']}'")
 
# Preview de lo que se va a ejecutar
print()
for a in archetypes_to_run:
    print(
        f"   → {a['source_name']:<25} | "
        f"format={a['file_format']:<6} | "
        f"target={a['target_catalog']}.{a['target_schema']}.{a['target_table']}"
    )


2026-05-27 04:05:20,171 [INFO] Leyendo metadata desde: /Volumes/fintech_finpay/default/vol_config/ingestion_archetypes.json
2026-05-27 04:05:20,802 [INFO] Arquetipos — total: 3 | activos: 3 | inactivos: 0


▶ Modo ALL — procesando 3 fuente(s) activa(s)

   → transactions              | format=csv    | target=fintech_finpay.bronze.transactions
   → merchants                 | format=json   | target=fintech_finpay.bronze.merchants
   → users                     | format=txt    | target=fintech_finpay.bronze.users


In [0]:
results = []
 
for archetype in archetypes_to_run:
    result = ingest_source(spark, archetype)
    results.append(result)

2026-05-27 04:05:20,989 [INFO] ============================================================
2026-05-27 04:05:20,990 [INFO] ▶ [transactions] → fintech_finpay.bronze.transactions
2026-05-27 04:05:20,991 [INFO]   [SCHEMA] transactions — 10 campos | infer_column_types=false ✓
2026-05-27 04:05:20,991 [INFO]   [CSV/TXT] path=/Volumes/fintech_finpay/default/vol_landing/transactions/ | pattern=transactions_*.csv | delimiter=',' | header=true
2026-05-27 04:05:20,992 [INFO]   [OBS] Columnas de observabilidad añadidas ✓
2026-05-27 04:05:21,236 [INFO]   [WRITE] → fintech_finpay.bronze.transactions | mode=append (streaming) | checkpoint=/Volumes/fintech_finpay/default/vol_metadata/checkpoints/transactions/
2026-05-27 04:05:27,919 [INFO]   [PROPS] Table properties aplicadas: ['delta.autoOptimize.optimizeWrite', 'delta.autoOptimize.autoCompact']
2026-05-27 04:05:27,920 [INFO] ✅ [transactions] completado — 51,150 registros
2026-05-27 04:05:27,920 [INFO] ================================================

In [0]:
success_count = sum(1 for r in results if r["status"] == "SUCCESS")
failed_count  = sum(1 for r in results if r["status"] == "FAILED")
total_records = sum(r["records"] for r in results)
total_time    = sum(r["duration_sec"] for r in results)
 
print()
print("=" * 75)
print("  📊  REPORTE FINAL — PIPELINE INGESTA BRONZE")
print(f"  🎯  Modo: {'ALL' if SOURCE_FILTER == 'all' else f'SINGLE - solo carga la fuente: {SOURCE_FILTER}'}")
print("=" * 75)
print(f"  Fuentes procesadas  : {len(results)}")
print(f"  ✅ Exitosas          : {success_count}")
print(f"  ❌ Fallidas          : {failed_count}")
print(f"  Registros totales   : {total_records:,}")
print(f"  Tiempo total        : {total_time:.2f}s")
print("=" * 75)
 
for r in results:
    icon  = "✅" if r["status"] == "SUCCESS" else "❌"
    error = f"\n      ↳ ERROR: {r['error']}" if r["error"] else ""
    print(
        f"  {icon}  {r['source_name']:<22}"
        f" | format={r['file_format']:<5}"
        f" | mode={r['write_mode']:<9}"
        f" | dedup={r['dedup_mode']:<16}"
        f" | {r['records']:>8,} rows"
        f" | {r['duration_sec']:>6.2f}s"
        f"{error}"
    )
 
print("=" * 75)


  📊  REPORTE FINAL — PIPELINE INGESTA BRONZE
  🎯  Modo: ALL
  Fuentes procesadas  : 3
  ✅ Exitosas          : 3
  ❌ Fallidas          : 0
  Registros totales   : 61,547
  Tiempo total        : 21.12s
  ✅  transactions           | format=csv   | mode=append (availableNow) | dedup=none             |   51,150 rows |   6.93s
  ✅  merchants              | format=json  | mode=append (availableNow) | dedup=none             |      500 rows |   7.07s
  ✅  users                  | format=txt   | mode=append (availableNow) | dedup=none             |    9,897 rows |   7.12s


In [0]:
from datetime import datetime, timezone
from pyspark.sql import Row
import uuid
PIPELINE_RUN_ID = str(uuid.uuid4())
OBSERVABILITY_TABLE = "fintech_finpay.observability.pipeline_event_log"

try:
    NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
except Exception:
    NOTEBOOK_PATH = "unknown"


def count_bad_records(spark, bad_records_path: str) -> int:
    """
    Cuenta los registros en la carpeta bad_records de una fuente.
    Retorna 0 si la carpeta no existe o está vacía.
    """
    try:
        files = dbutils.fs.ls(bad_records_path)
        if not files:
            return 0
        # Leer los JSON de bad records y contar filas
        return spark.read.format("json").load(bad_records_path).count()
    except Exception:
        return 0
 
 
event_ts   = datetime.now(timezone.utc)
event_date = event_ts.date()
 
event_rows = []
 
for r in results:
    # Buscar el arquetipo correspondiente para obtener bad_records_path
    archetype = next(
        (a for a in archetypes_to_run if a["source_name"] == r["source_name"]),
        {}
    )
 
    bad_records_path = archetype.get("bad_records_path", "")
    records_failed   = count_bad_records(spark, bad_records_path) if bad_records_path else 0
 
    # Determinar la capa desde target_schema del arquetipo
    layer = archetype.get("target_schema", "bronze")
 
    event_rows.append(
        Row(
            event_id          = str(uuid.uuid4()),
            pipeline_run_id   = PIPELINE_RUN_ID,
            event_ts          = event_ts,
            event_date        = event_date,
            layer             = layer,
            source_name       = r["source_name"],
            target_table      = r["target_table"],
            file_format       = r.get("file_format", ""),
            status            = r["status"],
            records_processed = int(r["records"]),
            records_failed    = int(records_failed),
            error_message     = r.get("error"),
            duration_sec      = float(r["duration_sec"]),
            write_mode        = str(r.get("write_mode", "")),
            dedup_mode        = str(r.get("dedup_mode", "")),
            schema_version    = int(archetype.get("schema_version", 1)),
            source_filter     = SOURCE_FILTER,
            notebook_path     = NOTEBOOK_PATH,
        )
    )
 
# Schema explícito — necesario porque error_message puede ser None
# y Spark no puede inferir el tipo de campos nulos sin schema declarado.
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, DoubleType, IntegerType, DateType, TimestampType
)
 
EVENT_LOG_SCHEMA = StructType([
    StructField("event_id",          StringType(),    nullable=False),
    StructField("pipeline_run_id",   StringType(),    nullable=False),
    StructField("event_ts",          TimestampType(), nullable=False),
    StructField("event_date",        DateType(),      nullable=False),
    StructField("layer",             StringType(),    nullable=True),
    StructField("source_name",       StringType(),    nullable=True),
    StructField("target_table",      StringType(),    nullable=True),
    StructField("file_format",       StringType(),    nullable=True),
    StructField("status",            StringType(),    nullable=True),
    StructField("records_processed", LongType(),      nullable=True),
    StructField("records_failed",    LongType(),      nullable=True),
    StructField("error_message",     StringType(),    nullable=True),
    StructField("duration_sec",      DoubleType(),    nullable=True),
    StructField("write_mode",        StringType(),    nullable=True),
    StructField("dedup_mode",        StringType(),    nullable=True),
    StructField("schema_version",    IntegerType(),   nullable=True),
    StructField("source_filter",     StringType(),    nullable=True),
    StructField("notebook_path",     StringType(),    nullable=True),
])
 
df_events = spark.createDataFrame(event_rows, schema=EVENT_LOG_SCHEMA)
 
(
    df_events.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(OBSERVABILITY_TABLE)
)
 
print(f"\n✅ Event log persistido en {OBSERVABILITY_TABLE}")
print(f"   pipeline_run_id : {PIPELINE_RUN_ID}")
print(f"   Eventos escritos: {len(event_rows)}")



✅ Event log persistido en fintech_finpay.observability.pipeline_event_log
   pipeline_run_id : 777d11fd-4a7a-4a13-ab76-a4ac6ffdf633
   Eventos escritos: 3


In [0]:
if failed_count > 0:
    raise RuntimeError(
        f"Pipeline Bronze finalizado con {failed_count} fuente(s) fallida(s). "
        f"Revisa los logs de cada fuente en las celdas anteriores."
    )